In [ ]:
#!/usr/bin/env python3
"""
Separate script to evaluate only ZNE and ZNE+TOPAZ.

TOPAZ logic is unchanged.

Fixes:
- Syntax bracket in path_scores.
- Memory: reduced 8-qubit simulator for optimisation.
- Transpile: CouplingMap object.
- Qubit count: transpile on reduced 8-qubit target & use _sim_8.
- ZNE: manual global circuit folding + Richardson extrapolation.
- ZNE evaluation: use direct noisy density-matrix simulation instead of
  qiskit_aer.primitives.Estimator, to avoid API/version issues and to
  preserve folded circuits exactly.
"""

import time
import numpy as np
import scipy.linalg as la
import itertools
import networkx as nx
import warnings
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit.transpiler import CouplingMap

warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
N_QUBITS = 8
DIM = 2 ** N_QUBITS
MAX_ITERATIONS = 40
REG_LAMBDA = 1e-3
RUN_SEED = 0

# ── Helper matrices ───────────────────────────────────────────────────────────
_I = np.eye(2, dtype=complex)
_X = np.array([[0, 1], [1, 0]], dtype=complex)
_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
_Z = np.array([[1, 0], [0, -1]], dtype=complex)

_PAIRS_2Q = {
    'XX': np.kron(_X, _X), 'XY': np.kron(_X, _Y), 'XZ': np.kron(_X, _Z),
    'YX': np.kron(_Y, _X), 'YY': np.kron(_Y, _Y), 'YZ': np.kron(_Y, _Z),
    'ZX': np.kron(_Z, _X), 'ZY': np.kron(_Z, _Y), 'ZZ': np.kron(_Z, _Z),
}
_NN_PAULI_NAMES = ['XX', 'YY', 'ZZ']
N_PAIRS = N_QUBITS - 1
N_TERMS = N_PAIRS * len(_NN_PAULI_NAMES)   # 21
N_PARAMS = 2 * N_TERMS                     # 42

_TERM_INFO = []
for _pi in range(N_PAIRS):
    for _pn in _NN_PAULI_NAMES:
        _TERM_INFO.append({
            'P2q': _PAIRS_2Q[_pn],
            'numpy_i': _pi,
            'numpy_j': _pi + 1,
            'qk_qa': N_QUBITS - 2 - _pi,
            'qk_qb': N_QUBITS - 1 - _pi,
        })

_PERM = [int(format(i, f'0{N_QUBITS}b')[::-1], 2) for i in range(DIM)]

# ── FakeFez backend setup ────────────────────────────────────────────────────
print("Initializing FakeFez (156-qubit IBM device noise emulator)...")
_fake_backend = FakeFez()
_noise_model = NoiseModel.from_backend(_fake_backend)
_sim = AerSimulator(noise_model=_noise_model)
print("Ready: FakeFez noise simulator successfully configured.")

# ── Find the noisiest 8-qubit path ───────────────────────────────────────────
print("\nScanning coupling map to locate the Noisiest 8-qubit path...")
props = _fake_backend.properties()
cmap = _fake_backend.configuration().coupling_map
basis_gates = _fake_backend.configuration().basis_gates
_two_q_gate = [g for g in basis_gates if g in ["cx", "cz", "ecr"]][0]

G = nx.Graph()
for q1, q2 in cmap:
    try:
        err1 = props.gate_error(_two_q_gate, [q1, q2]) or 0.0
    except Exception:
        err1 = 0.0
    try:
        err2 = props.gate_error(_two_q_gate, [q2, q1]) or 0.0
    except Exception:
        err2 = 0.0
    err = (err1 + err2) / 2.0

    try:
        ro1 = props.readout_error(q1) or 0.0
    except Exception:
        ro1 = 0.0
    try:
        ro2 = props.readout_error(q2) or 0.0
    except Exception:
        ro2 = 0.0
    ro = (ro1 + ro2) / 2.0

    G.add_edge(q1, q2, weight=err + 0.1 * ro)

paths = []

def dfs(node, path):
    if len(path) == 8:
        paths.append(path)
        return
    for neighbor in G.neighbors(node):
        if neighbor not in path:
            dfs(neighbor, path + [neighbor])

for node in G.nodes:
    dfs(node, [node])

path_scores = [(sum(G[p[i]][p[i + 1]]['weight'] for i in range(7)), p) for p in paths]
path_scores.sort(key=lambda x: x[0])
noisiest_8 = path_scores[-1][1]
print(f"Target Layout (Noisiest 8): {noisiest_8} (Cumulative Score: {path_scores[-1][0]:.4f})")

# ── Build reduced 8-qubit simulator ──────────────────────────────────────────
def build_reduced_backend(backend, layout_8):
    phys_to_virt = {p: i for i, p in enumerate(layout_8)}
    full_cmap = backend.configuration().coupling_map
    reduced_edges = []
    for a, b in full_cmap:
        if a in layout_8 and b in layout_8:
            reduced_edges.append((phys_to_virt[a], phys_to_virt[b]))
    coupling_map = CouplingMap(reduced_edges)

    basis_gates = backend.configuration().basis_gates
    full_noise = NoiseModel.from_backend(backend)
    noise_model = NoiseModel(basis_gates=basis_gates)

    for phys_q in layout_8:
        virt_q = phys_to_virt[phys_q]
        try:
            ro_err = full_noise._local_readout_errors.get(phys_q)
            if ro_err:
                noise_model.add_readout_error(ro_err, [virt_q])
        except Exception:
            pass

    for gate_name in basis_gates:
        for qubits, error in full_noise._local_quantum_errors.get(gate_name, {}).items():
            new_qubits = tuple(phys_to_virt[q] for q in qubits if q in layout_8)
            if len(new_qubits) == len(qubits):
                noise_model.add_quantum_error(error, gate_name, new_qubits)

    return coupling_map, noise_model

_red_cmap, _red_noise = build_reduced_backend(_fake_backend, noisiest_8)
_red_basis = _fake_backend.configuration().basis_gates
_sim_8 = AerSimulator(noise_model=_red_noise, method='density_matrix')

# ── Core helper functions ────────────────────────────────────────────────────

def _qiskit_dm_to_numpy(rho_q):
    return rho_q[np.ix_(_PERM, _PERM)]

def _apply_gate(U_2q, psi_tensor, qi, qj):
    N = psi_tensor.ndim
    axes = [qi, qj] + [k for k in range(N) if k != qi and k != qj]
    psi = np.transpose(psi_tensor, axes).reshape(4, -1)
    psi = (U_2q @ psi).reshape((2, 2) + (2,) * (N - 2))
    return np.transpose(psi, np.argsort(axes))

def compute_ideal_state(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau = params[N_TERMS:]

    psi = np.zeros((2,) * N_QUBITS, dtype=complex)
    psi[(0,) * N_QUBITS] = 1.0

    for j, t in enumerate(_TERM_INFO):
        psi = _apply_gate(
            la.expm(-1j * rho_n[j] * tau[j] * t['P2q']),
            psi,
            t['numpy_i'],
            t['numpy_j']
        )
    return psi.flatten()

def _upte_reg_penalty(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau = params[N_TERMS:]
    total = 0.0

    for _pi in range(N_PAIRS):
        B = np.zeros((4, 4), dtype=complex)
        for _pp, _pn in enumerate(_NN_PAULI_NAMES):
            j = _pi * len(_NN_PAULI_NAMES) + _pp
            B += rho_n[j] * la.expm(-1j * rho_n[j] * tau[j] * _PAIRS_2Q[_pn])
        dev = B.conj().T @ B - np.eye(4, dtype=complex)
        total += np.real(np.trace(dev.conj().T @ dev))

    return total

def apply_fake_backend_noise(params, layout):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau = params[N_TERMS:]

    qc = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(
            la.expm(-1j * rho_n[j] * tau[j] * t['P2q']),
            [t['qk_qa'], t['qk_qb']]
        )

    qc_t = transpile(
        qc,
        coupling_map=_red_cmap,
        basis_gates=_red_basis,
        initial_layout=list(range(N_QUBITS)),
        optimization_level=1
    )
    qc_t.save_density_matrix()
    result = _sim_8.run(qc_t, shots=1).result()
    return _qiskit_dm_to_numpy(np.array(result.data(0)['density_matrix']))

def noise_aware_objective(params, psi_target, layout):
    psi_ideal = compute_ideal_state(params)
    rho_noisy = apply_fake_backend_noise(params, layout)
    ideal_fid = float(np.abs(np.vdot(psi_target, psi_ideal)) ** 2)
    noisy_fid = float(np.clip(np.real(psi_target.conj() @ rho_noisy @ psi_target), 0, 1))
    loss = 1.0 - noisy_fid + REG_LAMBDA * _upte_reg_penalty(params)
    return loss, noisy_fid, ideal_fid, rho_noisy

class MMAOptimizer:
    def __init__(self, n, move_limit=0.4, gamma=0.5):
        self.n = n
        self.move_limit = move_limit
        self.gamma = gamma
        self.L = None
        self.U = None
        self.prev_params = None
        self.prev_loss = None

    def init(self, p0, delta=0.6):
        self.L = p0 - delta
        self.U = p0 + delta
        self.prev_params = p0.copy()

    def step(self, x, grad):
        x_new = np.zeros_like(x)
        for i in range(self.n):
            g, xi, Li, Ui = grad[i], x[i], self.L[i], self.U[i]
            pi = abs(g) * (Ui - xi) ** 2 if g < 0 else 0.0
            qi = abs(g) * (xi - Li) ** 2 if g >= 0 else 0.0
            dn = pi / (Ui - xi + 1e-12) ** 2 + qi / (xi - Li + 1e-12) ** 2

            if dn > 1e-12:
                x_new[i] = xi + (
                    pi / (Ui - xi + 1e-12) - qi / (xi - Li + 1e-12)
                ) / dn
            else:
                x_new[i] = xi

            x_new[i] = np.clip(
                x_new[i],
                max(xi - self.move_limit, Li + 1e-6),
                min(xi + self.move_limit, Ui - 1e-6)
            )
        return x_new

    def update(self, x, loss):
        good = self.prev_loss is None or loss < self.prev_loss - 1e-8
        s = 1.2 / self.gamma if good else self.gamma
        self.L = x - s * (x - self.L)
        self.U = x + s * (self.U - x)
        self.L = np.minimum(self.L, self.U - 1e-4)
        self.U = np.maximum(self.U, self.L + 1e-4)
        self.prev_params = x.copy()
        self.prev_loss = loss

_prev_grad = None

def hybrid_gradient(params, psi_target, layout):
    global _prev_grad
    grad = np.zeros_like(params)

    for i in range(N_TERMS):
        pp = np.clip(params.copy(), 1e-6, None)
        pp[i] += 1e-4
        pm = np.clip(params.copy(), 1e-6, None)
        pm[i] -= 1e-4
        grad[i] = (
            noise_aware_objective(pp, psi_target, layout)[0]
            - noise_aware_objective(pm, psi_target, layout)[0]
        ) / 2e-4

    for i in range(N_TERMS, N_PARAMS):
        pp = params.copy()
        pp[i] += np.pi / 4
        pm = params.copy()
        pm[i] -= np.pi / 4
        grad[i] = (
            noise_aware_objective(pp, psi_target, layout)[0]
            - noise_aware_objective(pm, psi_target, layout)[0]
        ) / (np.pi / 2)

    if _prev_grad is None:
        _prev_grad = grad.copy()
        return grad

    g = 0.4 * grad + 0.6 * _prev_grad
    _prev_grad = g.copy()
    return g

def run_dual_mma(init_params, psi_target, layout, max_iter=MAX_ITERATIONS):
    global _prev_grad
    _prev_grad = None

    mma_r = MMAOptimizer(N_TERMS, move_limit=0.2)
    mma_r.init(init_params[:N_TERMS], delta=0.4)

    mma_t = MMAOptimizer(N_TERMS, move_limit=0.6)
    mma_t.init(init_params[N_TERMS:], delta=0.8)

    cur = init_params.copy()
    cur_loss, cur_fid, cur_ifid, _ = noise_aware_objective(cur, psi_target, layout)
    best_fid, best_params, stag = cur_fid, cur.copy(), 0

    for it in range(max_iter):
        grad = hybrid_gradient(cur, psi_target, layout)

        new = np.concatenate([
            mma_r.step(cur[:N_TERMS], grad[:N_TERMS]),
            mma_t.step(cur[N_TERMS:], grad[N_TERMS:])
        ])

        new_loss, new_fid, _, _ = noise_aware_objective(new, psi_target, layout)
        delta = new_fid - cur_fid

        if new_loss < cur_loss - 1e-6 or delta > -1e-5:
            cur, cur_loss, cur_fid = new, new_loss, new_fid
            if cur_fid > best_fid:
                best_fid, best_params, stag = cur_fid, cur.copy(), 0
            else:
                stag += 1

            mma_r.move_limit = min(
                0.4,
                mma_r.move_limit * (
                    1.4 if delta > 0.01 else
                    1.2 if delta > 0.001 else
                    0.9
                )
            )
        else:
            mma_r.move_limit = max(0.02, mma_r.move_limit * 0.8)
            stag += 1

        mma_t.move_limit = mma_r.move_limit * 2.0
        mma_r.update(cur[:N_TERMS], cur_loss)
        mma_t.update(cur[N_TERMS:], cur_loss)

        if (it + 1) % 5 == 0 or it == max_iter - 1:
            print(f"    Iter {it+1:2d}: NoisyFid={cur_fid:.4f} (Δ={delta:+.4f})")

        if stag > 15:
            print(f"    Optimisation Stagnated at iter {it+1}.")
            break

    print(f"  Optimisation Complete. Best NoisyFid: {best_fid:.4f}")
    return best_params

# ── Target state builder ─────────────────────────────────────────────────────
def build_target(seed=None):
    rng = np.random.default_rng(seed)
    terms = []
    for i in range(N_QUBITS - 1):
        for p in ['XX', 'YY', 'ZZ']:
            s = ['I'] * N_QUBITS
            s[i] = p[0]
            s[i + 1] = p[1]
            terms.append(''.join(s))
    return la.expm(
        -1j * SparsePauliOp(
            terms,
            coeffs=rng.uniform(-0.5, 0.5, len(terms))
        ).to_matrix(sparse=False)
    )

# ── Circuit builder ──────────────────────────────────────────────────────────
def build_circuit(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau = params[N_TERMS:]

    qc = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(
            la.expm(-1j * rho_n[j] * tau[j] * t['P2q']),
            [t['qk_qa'], t['qk_qb']]
        )
    return qc

# ── ZNE helpers ──────────────────────────────────────────────────────────────
def fold_circuit_global(qc_transpiled, scale_factor):
    """
    Global unitary folding:
        C_λ = C · (C† · C)^k,  where λ = 2k + 1.
    """
    if scale_factor == 1:
        return qc_transpiled.copy()

    n_folds = (scale_factor - 1) // 2
    qc_inv = qc_transpiled.inverse()
    folded = qc_transpiled.copy()

    for _ in range(n_folds):
        folded = folded.compose(qc_inv)
        folded = folded.compose(qc_transpiled)

    return folded

def richardson_extrapolate(scale_factors, expectations):
    """
    Polynomial Richardson extrapolation to scale_factor -> 0.
    For [1,3,5], this is a quadratic fit.
    """
    degree = len(scale_factors) - 1
    poly = np.polyfit(scale_factors, expectations, deg=degree)
    return float(np.polyval(poly, 0))

def eval_state_fidelity_exact_dm(qc_transpiled, psi_target):
    """
    Evaluate <psi_target| rho |psi_target> exactly from the noisy density matrix.
    qc_transpiled should be the ansatz circuit only (no U_target† appended).
    """
    qc_dm = qc_transpiled.copy()
    qc_dm.save_density_matrix()

    result = _sim_8.run(qc_dm, shots=1).result()
    rho = np.array(result.data(0)['density_matrix'])
    rho = _qiskit_dm_to_numpy(rho)

    fid = np.real(psi_target.conj() @ rho @ psi_target)
    return float(np.clip(fid, 0.0, 1.0))

# ── Fidelity evaluator (with manual ZNE) ─────────────────────────────────────
def eval_fidelity(qc_ansatz, layout, use_zne=False):
    """
    Evaluate fidelity of qc_ansatz against psi_target.

    Matches TOPAZ's internal objective: F = <psi_target| rho_noisy |psi_target>
    No U_target† is appended — direct state-overlap fidelity only.

    If use_zne=True:
      1. Transpile the ansatz onto the reduced 8-qubit backend
      2. Apply global folding at scale factors [1, 3, 5]
      3. Evaluate <psi_target|rho|psi_target> for each folded circuit
      4. Richardson extrapolate to λ=0 (zero noise)
    """
    qc_t = transpile(
        qc_ansatz,
        coupling_map=_red_cmap,
        basis_gates=_red_basis,
        initial_layout=list(range(N_QUBITS)),
        optimization_level=1
    )

    if not use_zne:
        return eval_state_fidelity_exact_dm(qc_t, psi_target)

    scale_factors = [1, 3, 5]
    expectations = []

    for sf in scale_factors:
        qc_folded = fold_circuit_global(qc_t, sf)
        val = eval_state_fidelity_exact_dm(qc_folded, psi_target)
        expectations.append(val)

    zne_val = richardson_extrapolate(scale_factors, expectations)

    print(f"    ZNE raw expectations λ={scale_factors}: "
          f"{[f'{e:.5f}' for e in expectations]}")
    print(f"    ZNE extrapolated -> {zne_val:.5f}")

    return float(np.clip(zne_val, 0.0, 1.0))

# ── Main execution ───────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "=" * 70)
    print(f"RUNNING SINGLE SEED ({RUN_SEED}) BENCHMARK ON NOISIEST 8 LAYOUT")
    print("=" * 70)

    U_target = build_target(seed=0)
    psi_0 = np.zeros(DIM, dtype=complex)
    psi_0[0] = 1.0
    psi_target = U_target @ psi_0

    rng = np.random.default_rng(RUN_SEED + 42)
    init_params = np.concatenate([
        rng.uniform(0.1, 0.4, N_TERMS),
        rng.uniform(0.1, np.pi, N_TERMS)
    ])

    qc_baseline = build_circuit(init_params)

    print(f"\n[Step 1/3] Running TOPAZ Structural MMA Optimisation (Seed {RUN_SEED})...")
    t0 = time.time()
    best_params = run_dual_mma(init_params, psi_target, noisiest_8, MAX_ITERATIONS)
    print(f"TOPAZ MMA optimisation completed in {time.time() - t0:.1f}s.")
    qc_topaz = build_circuit(best_params)

    print("\n[Step 2/3] Evaluating noisy baselines (no ZNE)...\n")
    fid_noisy_baseline = eval_fidelity(qc_baseline, noisiest_8, use_zne=False)
    print(f"  Noisy baseline (random init) : {fid_noisy_baseline:.4f}")
    fid_noisy_topaz = eval_fidelity(qc_topaz, noisiest_8, use_zne=False)
    print(f"  Noisy TOPAZ (optimised)      : {fid_noisy_topaz:.4f}")

    print("\n[Step 3/3] Running ZNE-only and ZNE+TOPAZ evaluations...\n")

    print("  ── ZNE alone (random-init circuit) ──")
    fid_zne_alone = eval_fidelity(qc_baseline, noisiest_8, use_zne=True)
    print(f"  ZNE alone fidelity: {fid_zne_alone:.4f}\n")

    print("  ── TOPAZ + ZNE (optimised circuit) ──")
    fid_topaz_zne = eval_fidelity(qc_topaz, noisiest_8, use_zne=True)
    print(f"  TOPAZ + ZNE fidelity: {fid_topaz_zne:.4f}")

    print("\n" + "=" * 70)
    print("  Summary")
    print("-" * 70)
    print(f"  Noisy baseline (no mitigation) : {fid_noisy_baseline:.4f}")
    print(f"  Noisy TOPAZ only               : {fid_noisy_topaz:.4f}")
    print(f"  ZNE alone                      : {fid_zne_alone:.4f}")
    print(f"  TOPAZ + ZNE (optimised)        : {fid_topaz_zne:.4f}")
    print("=" * 70)

[23:34:15] Fake backend: fake_fez
[23:34:15] Extracting paths from fake backend...
[23:34:16] Noisiest: [33, 32, 31, 30, 29, 28, 27, 17]
[23:34:16] Quietest: [125, 124, 123, 136, 143, 142, 141, 140]

[23:34:20] === TOPAZ Simulation ===

[23:34:20] Target: Trotter PTE circuit, seed=0
[23:34:20] Baseline: same ansatz structure, random params (seed=42)
[23:34:20] TOPAZ: MMA-optimised params on each path's noise model


[23:34:20] PATH: NOISIEST  [33, 32, 31, 30, 29, 28, 27, 17]

[23:34:20] [1/3] Baseline (random params, same ansatz structure)
  Baseline DM:  0.0106
[23:34:20] [base_NOISIEST] DFE (2000 shots)
[23:34:21] [base_NOISIEST] 200/2000  Fid=0.00500
[23:34:21] [base_NOISIEST] 400/2000  Fid=0.00250
[23:34:21] [base_NOISIEST] 600/2000  Fid=0.00500
[23:34:21] [base_NOISIEST] 800/2000  Fid=0.00875
[23:34:21] [base_NOISIEST] 1000/2000  Fid=0.00800
[23:34:21] [base_NOISIEST] 1200/2000  Fid=0.00833
[23:34:22] [base_NOISIEST] 1400/2000  Fid=0.00786
[23:34:22] [base_NOISIEST] 1600/2000  Fid

KeyboardInterrupt: 